In [27]:
import torch 
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, num_channels, use1x1conv = False, strides = 1):
        super(Residual,self).__init__()
        self.conv1 = nn.Conv2d(in_channels, num_channels, kernel_size = 3, padding = 1, stride = strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size = 3, padding = 1, stride = 1)
        self.conv3 = None
        if use1x1conv :
            self.conv3 = nn.Conv2d(in_channels, num_channels,1, stride = strides)
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, X):
        Y = nn.ReLU()(self.bn1(self.conv1(X)))   
        Y = self.bn2(self.conv2(Y))
        if self.conv3 != None:
            X = self.conv3(X)
        return nn.ReLU()(Y + X)



## [**ResNet Model**]

Hai tầng đầu tiên của ResNet giống hai tầng đầu tiên của GoogLeNet: tầng tích chập $7\times 7$  với 64 kênh đầu ra và sải bước 2, theo sau bởi tầng maxpool  $3\times 3$ với sải bước 2. Sự khác biệt là trong ResNet, mỗi tầng tích chập theo sau bởi tầng chuẩn hóa theo batch.




In [28]:
net = nn.Sequential()
net.add_module("7x7Conv64", nn.Conv2d(1,64,7,stride = 2,padding = 3))
net.add_module("BatchNorm", nn.BatchNorm2d(64))
net.add_module("Relu",nn.ReLU())
net.add_module("3x3MaxPool",nn.MaxPool2d(3,2,padding = 1))

In [29]:
def ResnetBlock(in_channels, num_channels, num_layers, first_block = False):
    blk = nn.Sequential()
    for i in range(num_layers):
        if i == 0 and not first_block:
            blk.add_module('Residual_{}'.format(i),Residual(in_channels, num_channels, use1x1conv = True, strides = 2))
        else: 
            blk.add_module('Residual_{}'.format(i),Residual(num_channels, num_channels))
    return blk
                          

In [30]:
net.add_module("ResNet_1",ResnetBlock(64,64,2,True))
net.add_module("ResNet_2",ResnetBlock(64,128,2))
net.add_module("ResNet_3",ResnetBlock(128,256,2))
net.add_module("ResNet_4",ResnetBlock(256,512,2))

net.add_module('GlobalAvr',nn.AdaptiveAvgPool2d((1, 1)))
net.add_module('Flatten',nn.Flatten())
net.add_module('FC',nn.Linear(512, 10))

In [32]:
X = torch.randn((1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape:\t', X.shape)

Conv2d output shape:	 torch.Size([1, 64, 112, 112])
BatchNorm2d output shape:	 torch.Size([1, 64, 112, 112])
ReLU output shape:	 torch.Size([1, 64, 112, 112])
MaxPool2d output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 64, 56, 56])
Sequential output shape:	 torch.Size([1, 128, 28, 28])
Sequential output shape:	 torch.Size([1, 256, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 512, 1, 1])
Flatten output shape:	 torch.Size([1, 512])
Linear output shape:	 torch.Size([1, 10])


In [33]:
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import random

In [34]:
epochs = 2

# Các tham số cần thiết cho quá trình traning.
learning_rate = 0.0001
batch_size = 128
display_step = 100

# Path lưu best model 
checkpoint = 'modelx.pth' # có thể để dạng *.pth

# device chúng ta dùng cuda
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# assert device == 'cuda' 

In [35]:
# Transform image 
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
    ])

# load dataset từ torchvision.datasets
train_dataset = datasets.MNIST('../data', train=True, download=True,transform=transform)
test_dataset = datasets.MNIST('../data', train=False,transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size)
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size)

In [36]:
# call model, set deivce
model = net.to(device)
# load lại pretrained model (nếu có)
try:
  model.load_state_dict(torch.load(checkpoint))
except:
  print("!!! Hãy train để có checkpoint file")

!!! Hãy train để có checkpoint file


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
best_val_loss = 999

for epoch in range(1,epochs):
    # Quá trình training 
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # Zero_grad
        output = model(data)
        loss = criterion(output,target)
        loss.backward() # backward
        optimizer.step() # update weights
        if batch_idx % display_step == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tTrain Loss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
    # Quá trình testing 
    model.eval()
    test_loss = 0
    correct = 0
    # set no grad cho quá trình testing
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            output = F.log_softmax(output,dim=1) # log softmax using F, chu y dim nhe
            test_loss += criterion(output,target)
            pred = output.argmax(dim=1,keepdim=True) # argmax để lấy predicted label, chú ý keepdim = True
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset) 
    if test_loss < best_val_loss:
      best_val_loss = test_loss
      torch.save(model.state_dict(), checkpoint)  # Lưu lại model
      print("***********    TEST_ACC = {}%    ***********".format(correct))

Train Epoch: 1 [0/60000 (0%)]	Train Loss: 2.445282


KeyboardInterrupt: 